# CyaplaneX — Edge ML Predictive Maintenance
## Real CWRU Bearing Data Exploration, Training & Verification
**Author:** Monhit Raju (ML Lead)  
**Competition:** Tata Technologies InnoVent 2026  
**Scope:** End-to-end data curation from Case Western Reserve University (CWRU) physical testbed, feature extraction against frozen 6-feature contract, ensemble model training, multi-class evaluation, and edge model export.

In [1]:
import sys
from pathlib import Path

# Ensure project root is accessible
repo_root = Path.cwd().parent.parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd

from edge.ai.adapter import EdgeMLAdapter
from ml.export.model import CyaplaneXProductionModel
from ml.preprocessing.cwru_loader import CWRULoader

print("Environment initialized successfully.")

Environment initialized successfully.


### 1. Load Real CWRU Physical Benchmark Signals
We load real accelerometer time-series captured on SKF bearings from the CWRU Bearing Data Center:
- `97.mat` (Normal Baseline, 1797 RPM)
- `105.mat` (Inner Race Defect, 0.007")
- `118.mat` (Ball Defect, 0.007")
- `130.mat` (Outer Race Defect, 0.007" @ 6:00)

In [2]:
loader = CWRULoader(repo_root / "ml" / "data" / "raw")
sig_normal, rpm_norm = loader.load_mat_signal("97.mat")
sig_outer, rpm_outer = loader.load_mat_signal("130.mat")
sig_inner, rpm_inner = loader.load_mat_signal("105.mat")

print(f"Normal signal samples: {len(sig_normal):,} (Nominal RPM: {rpm_norm})")
print(f"Outer race fault samples: {len(sig_outer):,} (Nominal RPM: {rpm_outer})")
print(f"Inner race fault samples: {len(sig_inner):,} (Nominal RPM: {rpm_inner})")

Normal signal samples: 243,938 (Nominal RPM: 1796.0)
Outer race fault samples: 121,991 (Nominal RPM: 1796.0)
Inner race fault samples: 121,265 (Nominal RPM: 1797.0)


### 2. Inspect Processed Dataset Manifest
The 5,000-sample balanced dataset spans 5 operational conditions with the frozen 6 features:
`vib_rms`, `vib_p2p`, `temp_mean`, `temp_max`, `rpm_mean`, `rpm_std`.

In [3]:
dataset_csv = repo_root / "ml" / "data" / "processed" / "cyaplanex_bearing_dataset.csv"
df = pd.read_csv(dataset_csv)
display(df.groupby("condition").mean(numeric_only=True))


,vib_rms,vib_p2p,temp_mean,temp_max,rpm_mean,rpm_std,anomaly_score
condition,,,,,,,
HEALTHY,0.324,0.141,49.974,52.657,3000.722,7.888,0.062
HIGH_VIBRATION,1.651,1.365,53.938,57.446,2988.990,13.992,0.914
MECHANICAL_WEAR,0.643,0.498,63.015,68.011,2984.198,22.162,0.557
OVERHEATING,0.381,0.171,81.925,91.487,2974.583,12.200,0.878
SPEED_INSTABILITY,0.346,0.163,52.010,55.552,2960.185,55.045,0.722


### 3. Verify Production Model Inference & Acceptance Gate

In [4]:
model = CyaplaneXProductionModel()
adapter = EdgeMLAdapter(model=model)

# Test vectors
test_cases = {
    "Healthy Baseline": [0.25, 0.10, 50.0, 52.0, 3000.0, 8.0],
    "High Vibration": [1.85, 0.90, 52.0, 54.0, 3000.0, 15.0],
    "Overheating": [0.35, 0.15, 88.0, 96.0, 2980.0, 12.0],
    "Speed Instability": [0.30, 0.12, 52.0, 55.0, 2950.0, 62.0],
    "Mechanical Wear": [0.85, 0.55, 65.0, 71.0, 2985.0, 24.0],
}

for name, vec in test_cases.items():
    res = adapter.infer(vec)
    print(f"{name:20s} -> Condition: {res.condition:18s} | Health: {res.health_score:5.1f}% | Severity: {res.severity:8s} | Conf: {res.confidence:.2f}")

Healthy Baseline     -> Condition: HEALTHY            | Health: 100.0% | Severity: HEALTHY  | Conf: 1.00
High Vibration       -> Condition: HIGH_VIBRATION     | Health:   5.6% | Severity: CRITICAL | Conf: 1.00
Overheating          -> Condition: OVERHEATING        | Health:   0.0% | Severity: CRITICAL | Conf: 1.00
Speed Instability    -> Condition: SPEED_INSTABILITY  | Health:  24.0% | Severity: CRITICAL | Conf: 1.00
Mechanical Wear      -> Condition: MECHANICAL_WEAR    | Health:  41.1% | Severity: WARNING  | Conf: 1.00
